# Notebook 06: Unsupervised Learning — Customer Segmentation & PCA

## Credit Card Customer Churn & Segmentation
**Objective:** Discover natural behavioral customer segments using K-Means clustering without using churn labels, evaluate cluster validity with Elbow & Silhouette analysis, project clusters onto 2D PCA space, and profile segment churn risks.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from src.models.train_clustering import train_clustering_and_pca
from src.utils.helpers import MODELS_DIR

meta = train_clustering_and_pca(optimal_k=4)
print("Customer Segmentation & PCA Pipeline Completed.")

## 1. Optimal K Selection: Elbow & Silhouette
We evaluate $k \in [2, 6]$ on standardized clustering features.

In [ ]:
k_eval_df = pd.DataFrame(meta["k_evaluation"])
fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(k_eval_df["k"], k_eval_df["inertia"], 'b-o', label="Inertia (Elbow)")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia", color='b')

ax2 = ax1.twinx()
ax2.plot(k_eval_df["k"], k_eval_df["silhouette_score"], 'r-s', label="Silhouette Score")
ax2.set_ylabel("Silhouette Score", color='r')

plt.title("K-Means Evaluation: Elbow Method and Silhouette Analysis")
plt.tight_layout()
plt.show()

## 2. Customer Segment Profiles & Churn Rates
Post-hoc profiling reveals dramatically different customer archetypes and churn risks:

In [ ]:
profiles_df = pd.DataFrame(meta["cluster_profiles"])
cols_to_show = ["cluster_id", "segment_name", "customer_count", "percentage_of_total", "churn_rate_pct", "avg_transaction_amt", "avg_transaction_count", "avg_utilization_ratio", "avg_contacts_count"]
profiles_df[cols_to_show]

## 3. PCA Dimensionality Reduction & 2D Cluster Map
Principal Component Analysis projects the 9-dimensional clustering space to 2 dimensions for visualization.
- PC1 explains **23.3%** of variance (driven by transaction volume and spend).
- PC2 explains **17.0%** of variance (driven by credit limit and utilization).

In [ ]:
scatter_df = pd.DataFrame(meta["pca"]["scatter_sample"])
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=scatter_df,
    x="pc1",
    y="pc2",
    hue="segment_name",
    palette=["#2563eb", "#10b981", "#f59e0b", "#ef4444"],
    alpha=0.7,
    s=40
)
plt.title("2D PCA Projection of Discovered Customer Segments")
plt.xlabel("Principal Component 1 (Transaction Momentum & Volume)")
plt.ylabel("Principal Component 2 (Credit Limit & Utilization)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Combined Customer Intelligence Layer
By combining supervised churn probability with unsupervised segment profiles, retention managers can deploy targeted playbooks:
- **Disengaged & Underutilized**: 26.9% churn rate -> Immediate outbound retention & fee adjustment.
- **High-Utilization Revolvers**: 8.1% churn rate -> Credit line extension and auto-pay discounts.
- **Accelerating Growth**: 5.4% churn rate -> Cross-sell mortgages and investment products.
- **High-Volume Spenders**: 9.6% churn rate -> Premium rewards and concierge perks.